# Análisis Exploratorio de Datos (EDA) - TransCarga S.A.S.

## Objetivo
Analizar los datasets de clientes, vehículos, combustible y rutas para identificar patrones, relaciones y estadísticas útiles que apoyen la optimización logística.

---

## 1. Importación de Librerías y Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Colores corporativos
COLORS = {
    'primary': '#1a365d',
    'secondary': '#2b6cb0',
    'success': '#38a169',
    'warning': '#dd6b20',
    'danger': '#e53e3e'
}

print("Librerías importadas correctamente")

In [ ]:
# Cargar todos los datasets
base_path = r'C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos'

# Datos procesados
clientes = pd.read_csv(f'{base_path}/processed/clientes_clean.csv')
vehiculos = pd.read_csv(f'{base_path}/processed/vehiculos_clean.csv')
combustible = pd.read_csv(f'{base_path}/processed/combustible_clean.csv')
divipola = pd.read_csv(f'{base_path}/processed/divipola_clean.csv')

# Matrices
matriz_dist = pd.read_csv(f'{base_path}/processed/matriz_distancias.csv')
matriz_tiempo = pd.read_csv(f'{base_path}/processed/matriz_tiempos.csv')

# Resultados de optimización
rutas = pd.read_csv(f'{base_path}/results/rutas_optimizadas.csv')

print(f"Datasets cargados:")
print(f"  - Clientes: {len(clientes)} registros")
print(f"  - Vehículos: {len(vehiculos)} registros")
print(f"  - Combustible: {len(combustible)} registros")
print(f"  - DIVIPOLA: {len(divipola)} registros")
print(f"  - Matriz distancias: {matriz_dist.shape}")
print(f"  - Rutas optimizadas: {len(rutas)} registros")

---
## 2. Análisis de Clientes

### 2.1 Resumen Estadístico

In [ ]:
# Información general del dataset
print("="*60)
print("INFORMACIÓN GENERAL - CLIENTES")
print("="*60)
print(f"\nTotal de clientes: {len(clientes)}")
print(f"Columnas: {clientes.columns.tolist()}")
print(f"\nValores nulos por columna:")
print(clientes.isnull().sum())
print(f"\nTipos de datos:")
print(clientes.dtypes)

In [ ]:
# Estadísticas descriptivas
print("\n" + "="*60)
print("ESTADÍSTICAS DESCRIPTIVAS - CLIENTES")
print("="*60)
clientes.describe()

### 2.2 Distribución por Departamento

In [ ]:
# Distribución de clientes por departamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
dept_counts = clientes['departamento'].value_counts()
colors = [COLORS['primary'], COLORS['secondary']]

axes[0].bar(dept_counts.index, dept_counts.values, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_title('Clientes por Departamento', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Departamento')
axes[0].set_ylabel('Número de Clientes')
axes[0].tick_params(axis='x', rotation=0)

# Agregar valores encima de las barras
for i, v in enumerate(dept_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold', fontsize=12)

# Gráfico de pastel
axes[1].pie(dept_counts.values, labels=dept_counts.index, autopct='%1.1f%%', 
            colors=colors, explode=[0.05, 0], shadow=True, startangle=90)
axes[1].set_title('Proporción de Clientes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_clientes_departamento.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Insights:")
print(f"  - {dept_counts.index[0]}: {dept_counts.values[0]} clientes ({dept_counts.values[0]/len(clientes)*100:.1f}%)")
print(f"  - {dept_counts.index[1]}: {dept_counts.values[1]} clientes ({dept_counts.values[1]/len(clientes)*100:.1f}%)")

### 2.3 Distribución por Municipio (Top 15)

In [ ]:
# Top 15 municipios con más clientes
muni_counts = clientes['municipio'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 7))

bars = ax.barh(muni_counts.index[::-1], muni_counts.values[::-1], 
               color=COLORS['secondary'], edgecolor='black', linewidth=1)

ax.set_title('Top 15 Municipios con Más Clientes', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de Clientes')
ax.set_ylabel('Municipio')

# Agregar valores
for bar, val in zip(bars, muni_counts.values[::-1]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, str(val),
            va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_clientes_municipios.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Top 5 municipios:")
for i, (muni, count) in enumerate(muni_counts.head(5).items(), 1):
    print(f"  {i}. {muni}: {count} clientes")

### 2.4 Análisis de Demanda

In [ ]:
# Distribución de demanda
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma
axes[0, 0].hist(clientes['demanda_kg'], bins=30, color=COLORS['primary'], 
                edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Histograma de Demanda', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Demanda (kg)')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].axvline(clientes['demanda_kg'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Media: {clientes["demanda_kg"].mean():.0f} kg')
axes[0, 0].legend()

# Boxplot
axes[0, 1].boxplot(clientes['demanda_kg'], vert=True, patch_artist=True,
                   boxprops=dict(facecolor=COLORS['secondary']))
axes[0, 1].set_title('Boxplot de Demanda', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Demanda (kg)')

# Demanda por departamento
clientes.groupby('departamento')['demanda_kg'].plot(kind='hist', ax=axes[1, 0], 
                                                      alpha=0.6, bins=20, edgecolor='black')
axes[1, 0].set_title('Demanda por Departamento', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Demanda (kg)')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].legend(clientes['departamento'].unique())

# Demanda acumulada
demanda_acum = clientes['demanda_kg'].sort_values().cumsum()
demanda_acum_pct = demanda_acum / demanda_acum.max() * 100
axes[1, 1].plot(range(len(demanda_acum)), demanda_acum_pct, color=COLORS['success'], linewidth=2)
axes[1, 1].fill_between(range(len(demanda_acum)), demanda_acum_pct, alpha=0.3, color=COLORS['success'])
axes[1, 1].set_title('Demanda Acumulada (Curva ABC)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Clientes ordenados por demanda')
axes[1, 1].set_ylabel('Demanda Acumulada (%)')
axes[1, 1].axhline(80, color='red', linestyle='--', alpha=0.7, label='80%')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_demanda_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()

# Estadísticas de demanda
print(f"\n📊 Estadísticas de Demanda:")
print(f"  - Media: {clientes['demanda_kg'].mean():.0f} kg")
print(f"  - Mediana: {clientes['demanda_kg'].median():.0f} kg")
print(f"  - Desviación estándar: {clientes['demanda_kg'].std():.0f} kg")
print(f"  - Mínimo: {clientes['demanda_kg'].min():.0f} kg")
print(f"  - Máximo: {clientes['demanda_kg'].max():.0f} kg")
print(f"  - Total: {clientes['demanda_kg'].sum():,.0f} kg")

### 2.5 Distribución Geográfica de Clientes

In [ ]:
# Scatter plot geográfico
fig, ax = plt.subplots(figsize=(12, 10))

# Plot por departamento
for dept, color in zip(clientes['departamento'].unique(), [COLORS['primary'], COLORS['secondary']]):
    subset = clientes[clientes['departamento'] == dept]
    ax.scatter(subset['longitud'], subset['latitud'], 
               c=color, label=dept, alpha=0.6, s=50, edgecolor='white', linewidth=0.5)

ax.set_title('Distribución Geográfica de Clientes', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.legend(title='Departamento')
ax.grid(True, alpha=0.3)

# Agregar referencia de bodegas
ax.annotate('Medellín\n(Bodega)', xy=(-75.56, 6.25), fontsize=10, 
            ha='center', color='red', fontweight='bold')
ax.annotate('Cali\n(Bodega)', xy(-76.52, 3.45), fontsize=10, 
            ha='center', color='red', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_clientes_geografico.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Coordenadas geográficas:")
print(f"  - Latitud: {clientes['latitud'].min():.2f} a {clientes['latitud'].max():.2f}")
print(f"  - Longitud: {clientes['longitud'].min():.2f} a {clientes['longitud'].max():.2f}")

---
## 3. Análisis de Vehículos

### 3.1 Resumen Estadístico

In [ ]:
print("="*60)
print("INFORMACIÓN GENERAL - VEHÍCULOS")
print("="*60)
print(f"\nTotal de vehículos: {len(vehiculos)}")
print(f"Columnas: {vehiculos.columns.tolist()}")
print(f"\nValores nulos por columna:")
print(vehiculos.isnull().sum())

### 3.2 Distribución por Tipo de Vehículo

In [ ]:
# Distribución por tipo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tipo_counts = vehiculos['tipo'].value_counts()
colors_tipos = [COLORS['primary'], COLORS['secondary'], COLORS['success'], COLORS['warning']]

# Gráfico de barras
axes[0].bar(tipo_counts.index, tipo_counts.values, color=colors_tipos[:len(tipo_counts)], 
            edgecolor='black', linewidth=1.2)
axes[0].set_title('Vehículos por Tipo', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tipo de Vehículo')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=45)

for i, v in enumerate(tipo_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold', fontsize=12)

# Gráfico de pastel
axes[1].pie(tipo_counts.values, labels=tipo_counts.index, autopct='%1.1f%%',
            colors=colors_tipos[:len(tipo_counts)], shadow=True, startangle=90)
axes[1].set_title('Proporción por Tipo', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_vehiculos_tipo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Distribución por tipo:")
for tipo, count in tipo_counts.items():
    print(f"  - {tipo}: {count} vehículos ({count/len(vehiculos)*100:.1f}%)")

### 3.3 Análisis de Capacidad

In [ ]:
# Capacidad por tipo de vehículo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de capacidad por tipo
vehiculos.boxplot(column='capacidad_kg', by='tipo', ax=axes[0])
axes[0].set_title('Capacidad por Tipo de Vehículo', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tipo de Vehículo')
axes[0].set_ylabel('Capacidad (kg)')
plt.suptitle('')  # Quitar título automático

# Histograma de capacidad
axes[1].hist(vehiculos['capacidad_kg'], bins=20, color=COLORS['success'], 
             edgecolor='black', alpha=0.7)
axes[1].set_title('Distribución de Capacidad', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Capacidad (kg)')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(vehiculos['capacidad_kg'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Media: {vehiculos["capacidad_kg"].mean():.0f} kg')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_vehiculos_capacidad.png', dpi=150, bbox_inches='tight')
plt.show()

# Capacidad total
capacidad_total = vehiculos['capacidad_kg'].sum()
print(f"\n📊 Estadísticas de Capacidad:")
print(f"  - Capacidad media: {vehiculos['capacidad_kg'].mean():,.0f} kg")
print(f"  - Capacidad mínima: {vehiculos['capacidad_kg'].min():,.0f} kg")
print(f"  - Capacidad máxima: {vehiculos['capacidad_kg'].max():,.0f} kg")
print(f"  - Capacidad total de la flota: {capacidad_total:,.0f} kg")

---
## 4. Análisis de Combustible

In [ ]:
print("="*60)
print("ANÁLISIS DE PRECIOS DE COMBUSTIBLE")
print("="*60)
print(f"\nTotal de registros: {len(combustible)}")

# Precios por departamento
precios_dept = combustible.groupby('departamento')['precio'].agg(['mean', 'min', 'max', 'count'])
precios_dept.columns = ['Media', 'Mínimo', 'Máximo', 'Estaciones']

print(f"\n📊 Precios por Departamento:")
print(precios_dept)

In [ ]:
# Visualización de precios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de precios
axes[0].hist(combustible['precio'], bins=30, color=COLORS['warning'], 
             edgecolor='black', alpha=0.7)
axes[0].set_title('Distribución de Precios de Combustible', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Precio (COP/galón)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(combustible['precio'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Media: ${combustible["precio"].mean():,.0f}')
axes[0].legend()

# Boxplot por departamento
combustible.boxplot(column='precio', by='departamento', ax=axes[1])
axes[1].set_title('Precios por Departamento', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Departamento')
axes[1].set_ylabel('Precio (COP/galón)')
plt.suptitle('')

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_combustible.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Estadísticas de Precios:")
print(f"  - Precio medio: ${combustible['precio'].mean():,.0f} COP/galón")
print(f"  - Precio mínimo: ${combustible['precio'].min():,.0f} COP/galón")
print(f"  - Precio máximo: ${combustible['precio'].max():,.0f} COP/galón")

---
## 5. Análisis de Matriz de Distancias

In [ ]:
print("="*60)
print("ANÁLISIS DE MATRIZ DE DISTANCIAS")
print("="*60)
print(f"\nDimensiones: {matriz_dist.shape}")

# Convertir a numpy para análisis
dist_matrix = matriz_dist.values

# Eliminar diagonales (distancia 0) para estadísticas
dist_sin_zero = dist_matrix[dist_matrix > 0]

print(f"\n📊 Estadísticas de Distancia:")
print(f"  - Distancia mínima: {dist_sin_zero.min():.1f} km")
print(f"  - Distancia máxima: {dist_sin_zero.max():.1f} km")
print(f"  - Distancia media: {dist_sin_zero.mean():.1f} km")
print(f"  - Distancia mediana: {np.median(dist_sin_zero):.1f} km")

In [ ]:
# Visualización de matriz de distancias
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap de matriz (primeros 20x20)
sns.heatmap(matriz_dist.iloc[:20, :20], cmap='YlOrRd', ax=axes[0], 
            cbar_kws={'label': 'Distancia (km)'})
axes[0].set_title('Matriz de Distancias (Primeros 20 nodos)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nodo destino')
axes[0].set_ylabel('Nodo origen')

# Histograma de distancias
axes[1].hist(dist_sin_zero, bins=50, color=COLORS['secondary'], 
             edgecolor='black', alpha=0.7)
axes[1].set_title('Distribución de Distancias', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Distancia (km)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_matriz_distancias.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Análisis de Rutas Optimizadas

In [ ]:
# Normalizar nombres de columnas
rutas = rutas.rename(columns={'orden': 'parada_num', 'demanda': 'carga_kg'})

print("="*60)
print("ANÁLISIS DE RUTAS OPTIMIZADAS")
print("="*60)
print(f"\nTotal de paradas: {len(rutas)}")
print(f"Vehículos utilizados: {rutas['vehiculo_id'].nunique()}")
print(f"\nColumnas: {rutas.columns.tolist()}")

In [ ]:
# Resumen por vehículo
resumen_vehiculos = rutas.groupby('vehiculo_id').agg({
    'parada_num': 'count',
    'carga_kg': 'sum'
}).reset_index()
resumen_vehiculos.columns = ['Vehículo', 'Paradas', 'Carga Total (kg)']

print(f"\n📊 Resumen por Vehículo:")
print(resumen_vehiculos)

In [ ]:
# Visualización de rutas por vehículo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Paradas por vehículo
axes[0].bar(resumen_vehiculos['Vehículo'].astype(str), 
            resumen_vehiculos['Paradas'], 
            color=COLORS['primary'], edgecolor='black')
axes[0].set_title('Paradas por Vehículo', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Vehículo')
axes[0].set_ylabel('Número de Paradas')

for i, v in enumerate(resumen_vehiculos['Paradas']):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# Carga por vehículo
axes[1].bar(resumen_vehiculos['Vehículo'].astype(str), 
            resumen_vehiculos['Carga Total (kg)'], 
            color=COLORS['success'], edgecolor='black')
axes[1].set_title('Carga Entregada por Vehículo', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Vehículo')
axes[1].set_ylabel('Carga (kg)')

for i, v in enumerate(resumen_vehiculos['Carga Total (kg)']):
    axes[1].text(i, v + 100, f'{v:,.0f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(f'{base_path}/../img/eda_rutas_vehiculos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Conclusiones del EDA

In [ ]:
print("="*70)
print("CONCLUSIONES DEL ANÁLISIS EXPLORATORIO DE DATOS")
print("="*70)

conclusiones = f"""
📊 RESUMEN EJECUTIVO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. CLIENTES (n={len(clientes)})
   • Distribución geográfica: {clientes['departamento'].value_counts().to_dict()}
   • Demanda media: {clientes['demanda_kg'].mean():.0f} kg por cliente
   • Demanda total: {clientes['demanda_kg'].sum():,.0f} kg
   • Concentración: Mayoría en {clientes['municipio'].value_counts().index[0]}

2. VEHÍCULOS (n={len(vehiculos)})
   • Tipos disponibles: {vehiculos['tipo'].value_counts().to_dict()}
   • Capacidad media: {vehiculos['capacidad_kg'].mean():,.0f} kg
   • Capacidad total: {vehiculos['capacidad_kg'].sum():,.0f} kg
   • Ratio demanda/capacidad: {clientes['demanda_kg'].sum()/vehiculos['capacidad_kg'].sum()*100:.1f}%

3. COMBUSTIBLE
   • Precio medio: ${combustible['precio'].mean():,.0f} COP/galón
   • Variación: ${combustible['precio'].std():,.0f} COP entre estaciones

4. OPTIMIZACIÓN
   • Vehículos utilizados: {rutas['vehiculo_id'].nunique()} de {len(vehiculos)}
   • Clientes atendidos: {len(rutas[rutas['carga_kg'] > 0])}
   • Carga entregada: {rutas['carga_kg'].sum():,.0f} kg

🎯 INSIGHTS CLAVE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• La flota tiene capacidad suficiente para atender toda la demanda
• Los clientes están concentrados en Antioquia y Valle del Cauca
• La demanda presenta distribución normal con algunos clientes de alta demanda
• El modelo de optimización logra atender 20 clientes con 5 vehículos

📈 RECOMENDACIONES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Implementar análisis de clustering para segmentar clientes
• Considerar restricciones de tiempo para mejorar el modelo
• Monitorear precios de combustible para ajustar costos
• Expandir cobertura a más clientes con el modelo escalado
"""

print(conclusiones)

---
## 8. Guardar Resumen del EDA

In [ ]:
# Crear resumen en JSON
import json

eda_summary = {
    'fecha_analisis': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
    'clientes': {
        'total': len(clientes),
        'por_departamento': clientes['departamento'].value_counts().to_dict(),
        'demanda_media_kg': round(clientes['demanda_kg'].mean(), 2),
        'demanda_total_kg': int(clientes['demanda_kg'].sum())
    },
    'vehiculos': {
        'total': len(vehiculos),
        'por_tipo': vehiculos['tipo'].value_counts().to_dict(),
        'capacidad_media_kg': round(vehiculos['capacidad_kg'].mean(), 2),
        'capacidad_total_kg': int(vehiculos['capacidad_kg'].sum())
    },
    'combustible': {
        'precio_medio_cop': round(combustible['precio'].mean(), 2),
        'precio_min_cop': int(combustible['precio'].min()),
        'precio_max_cop': int(combustible['precio'].max())
    },
    'optimizacion': {
        'vehiculos_utilizados': int(rutas['vehiculo_id'].nunique()),
        'clientes_atendidos': int(len(rutas[rutas['carga_kg'] > 0])),
        'carga_total_kg': int(rutas['carga_kg'].sum())
    }
}

# Guardar
with open(f'{base_path}/../documentacion/eda_summary.json', 'w', encoding='utf-8') as f:
    json.dump(eda_summary, f, indent=2, ensure_ascii=False)

print("✅ Resumen del EDA guardado en: documentacion/eda_summary.json")
print("\n📊 Gráficos guardados en: img/")
print("  - eda_clientes_departamento.png")
print("  - eda_clientes_municipios.png")
print("  - eda_demanda_distribucion.png")
print("  - eda_clientes_geografico.png")
print("  - eda_vehiculos_tipo.png")
print("  - eda_vehiculos_capacidad.png")
print("  - eda_combustible.png")
print("  - eda_matriz_distancias.png")
print("  - eda_rutas_vehiculos.png")

In [ ]:
print("\n" + "="*70)
print("✅ ANÁLISIS EDA COMPLETADO")
print("="*70)